# Whales Are Seen More on Weekends

Whales do not know what day it is. If the public record (Happywhale and iNaturalist) only recorded
whales, a Saturday would look like a Tuesday. This notebook measures how far it does not, and checks
the result against the hydrophones, which recorded on most days.

**Weekend ratio:** records on an average weekend day ÷ records on an average Tuesday-to-Thursday day,
that is (Sat + Sun) / 2 over (Tue + Wed + Thu) / 3, from `dim_date.is_weekend` and `is_midweek`. Monday
and Friday are left out, because long weekends and holidays blur them; the ratio against all five
weekdays is shown beside it. A source with no weekly rhythm scores about 1.

In [ ]:
%%sql
-- WHALES_WH runs the queries.
USE WAREHOUSE whales_wh

In [ ]:
%%sql
-- The star dbt builds lives in WHALES.MARTS.
USE SCHEMA whales.marts

## Public Sightings on Each Day of the Week
Public records inside the Monterey Bay box, counted by weekday.

In [ ]:
%%sql
SELECT
    dim_date.weekday_number,
    dim_date.weekday_name,
    COUNT(*) AS public_records
FROM fct_sighting
JOIN dim_location ON dim_location.location_key = fct_sighting.location_key
JOIN dim_date ON dim_date.date_key = fct_sighting.date_key
JOIN dim_dataset ON dim_dataset.dataset_key = fct_sighting.dataset_key
WHERE dim_location.in_bay
  AND dim_dataset.kind = 'public'
GROUP BY dim_date.weekday_number, dim_date.weekday_name
ORDER BY dim_date.weekday_number

## The Weekend Ratio for Every Kind of Source

In [ ]:
%%sql
SELECT
    dim_dataset.kind,
    COUNT(*) AS records,
    ROUND(
        (COUNT_IF(dim_date.is_weekend) / 2) / NULLIF(COUNT_IF(dim_date.is_midweek) / 3, 0),
        2
    ) AS weekend_ratio,
    ROUND(
        (COUNT_IF(dim_date.is_weekend) / 2) / NULLIF(COUNT_IF(NOT dim_date.is_weekend) / 5, 0),
        2
    ) AS weekend_ratio_against_all_weekdays
FROM fct_sighting
JOIN dim_location ON dim_location.location_key = fct_sighting.location_key
JOIN dim_date ON dim_date.date_key = fct_sighting.date_key
JOIN dim_dataset ON dim_dataset.dataset_key = fct_sighting.dataset_key
WHERE dim_location.in_bay
GROUP BY dim_dataset.kind
ORDER BY weekend_ratio DESC

## What Sits Behind the Survey and Other Ratios
The same ratio for each survey and "other" dataset in the bay, with the number of days it has records on.

In [ ]:
%%sql
SELECT
    dim_dataset.kind,
    dim_dataset.title,
    COUNT(*) AS records,
    COUNT(DISTINCT fct_sighting.date_key) AS days,
    ROUND(
        (COUNT_IF(dim_date.is_weekend) / 2) / NULLIF(COUNT_IF(dim_date.is_midweek) / 3, 0),
        2
    ) AS weekend_ratio
FROM fct_sighting
JOIN dim_location ON dim_location.location_key = fct_sighting.location_key
JOIN dim_date ON dim_date.date_key = fct_sighting.date_key
JOIN dim_dataset ON dim_dataset.dataset_key = fct_sighting.dataset_key
WHERE dim_location.in_bay
  AND dim_dataset.kind IN ('survey', 'other')
GROUP BY dim_dataset.kind, dim_dataset.title
ORDER BY dim_dataset.kind, records DESC

## How Sure Is the Public Ratio?
A few unusually busy weekends could carry the ratio on their own. So the public records are grouped
into Monday-to-Sunday weeks, the same number of weeks is drawn at random with replacement 2,000 times,
and the ratio is recomputed each time. The middle 95% of those ratios is the range.
`GENERATOR` makes the 2,000 replicates, and a `HASH` of the replicate and the draw picks each week, so
every run draws the same weeks and gives the same range. (`SAMPLE` cannot do this: it draws without
replacement, and Snowflake's seeded `RANDOM` does not repeat its draws from one run to the next.)

In [ ]:
%%sql
WITH weekly_counts AS (
    SELECT
        DATEADD(DAY, 1 - dim_date.weekday_number, dim_date.date) AS week_start,
        COUNT_IF(dim_date.is_weekend) AS weekend_records,
        COUNT_IF(dim_date.is_midweek) AS midweek_records
    FROM fct_sighting
    JOIN dim_location ON dim_location.location_key = fct_sighting.location_key
    JOIN dim_date ON dim_date.date_key = fct_sighting.date_key
    JOIN dim_dataset ON dim_dataset.dataset_key = fct_sighting.dataset_key
    WHERE dim_location.in_bay
      AND dim_dataset.kind = 'public'
    GROUP BY week_start
),

numbered_weeks AS (
    SELECT
        ROW_NUMBER() OVER (ORDER BY week_start) AS week_number,
        COUNT(*) OVER () AS total_weeks,
        weekend_records,
        midweek_records
    FROM weekly_counts
),

replicates AS (
    SELECT SEQ4() AS replicate
    FROM TABLE(GENERATOR(ROWCOUNT => 2000))
),

-- Each replicate draws as many weeks as there are, at random and with replacement.
resample_draws AS (
    SELECT
        replicates.replicate,
        1 + MOD(
            ABS(HASH(replicates.replicate, numbered_weeks.week_number, 20261016)),
            numbered_weeks.total_weeks
        ) AS week_number
    FROM replicates
    CROSS JOIN numbered_weeks
),

resampled_ratios AS (
    SELECT
        resample_draws.replicate,
        (SUM(numbered_weeks.weekend_records) / 2)
        / NULLIF(SUM(numbered_weeks.midweek_records) / 3, 0) AS weekend_ratio
    FROM resample_draws
    JOIN numbered_weeks ON numbered_weeks.week_number = resample_draws.week_number
    GROUP BY resample_draws.replicate
)

SELECT
    (SELECT MAX(total_weeks) FROM numbered_weeks) AS weeks,
    ROUND(PERCENTILE_CONT(0.025) WITHIN GROUP (ORDER BY weekend_ratio), 2) AS range_low,
    ROUND(PERCENTILE_CONT(0.975) WITHIN GROUP (ORDER BY weekend_ratio), 2) AS range_high
FROM resampled_ratios

## The Control: Did the Hydrophones Hear More Whales on Weekends?
NOAA's SanctSound hydrophones in the bay recorded on most days from November 2018 to June 2021
(`in_hydrophone_window`). Their record counts mix very different instruments, so this counts **days
with at least one detection** and applies the same ratio to days.

In [ ]:
%%sql
WITH window_days AS (
    SELECT
        dim_date.date_key,
        dim_date.is_weekend,
        dim_date.is_midweek
    FROM dim_date
    WHERE dim_date.in_hydrophone_window
),

heard_days AS (
    SELECT DISTINCT fct_sighting.date_key
    FROM fct_sighting
    JOIN dim_location ON dim_location.location_key = fct_sighting.location_key
    JOIN dim_dataset ON dim_dataset.dataset_key = fct_sighting.dataset_key
    WHERE dim_location.in_bay
      AND dim_dataset.kind = 'hydrophone'
)

SELECT
    COUNT(*) AS days_recording,
    COUNT(heard_days.date_key) AS days_heard,
    ROUND(days_heard / days_recording, 2) AS share_of_days_heard,
    ROUND(
        (COUNT_IF(window_days.is_weekend AND heard_days.date_key IS NOT NULL)
            / COUNT_IF(window_days.is_weekend))
        / NULLIF(
            COUNT_IF(window_days.is_midweek AND heard_days.date_key IS NOT NULL)
                / COUNT_IF(window_days.is_midweek),
            0
        ),
        2
    ) AS detection_day_ratio
FROM window_days
LEFT JOIN heard_days ON heard_days.date_key = window_days.date_key

## How Sure Is the Hydrophone Ratio?
The same resampling as for the public ratio: the weeks of the window drawn at random with replacement
2,000 times, and the share of weekend days heard over the share of midweek days heard each time.

In [ ]:
%%sql
WITH heard_days AS (
    SELECT DISTINCT fct_sighting.date_key
    FROM fct_sighting
    JOIN dim_location ON dim_location.location_key = fct_sighting.location_key
    JOIN dim_dataset ON dim_dataset.dataset_key = fct_sighting.dataset_key
    WHERE dim_location.in_bay
      AND dim_dataset.kind = 'hydrophone'
),

weekly_counts AS (
    SELECT
        DATEADD(DAY, 1 - dim_date.weekday_number, dim_date.date) AS week_start,
        COUNT_IF(dim_date.is_weekend) AS weekend_days,
        COUNT_IF(dim_date.is_weekend AND heard_days.date_key IS NOT NULL) AS weekend_days_heard,
        COUNT_IF(dim_date.is_midweek) AS midweek_days,
        COUNT_IF(dim_date.is_midweek AND heard_days.date_key IS NOT NULL) AS midweek_days_heard
    FROM dim_date
    LEFT JOIN heard_days ON heard_days.date_key = dim_date.date_key
    WHERE dim_date.in_hydrophone_window
    GROUP BY week_start
),

numbered_weeks AS (
    SELECT
        ROW_NUMBER() OVER (ORDER BY week_start) AS week_number,
        COUNT(*) OVER () AS total_weeks,
        weekend_days,
        weekend_days_heard,
        midweek_days,
        midweek_days_heard
    FROM weekly_counts
),

replicates AS (
    SELECT SEQ4() AS replicate
    FROM TABLE(GENERATOR(ROWCOUNT => 2000))
),

resample_draws AS (
    SELECT
        replicates.replicate,
        1 + MOD(
            ABS(HASH(replicates.replicate, numbered_weeks.week_number, 20261016)),
            numbered_weeks.total_weeks
        ) AS week_number
    FROM replicates
    CROSS JOIN numbered_weeks
),

resampled_ratios AS (
    SELECT
        resample_draws.replicate,
        (SUM(numbered_weeks.weekend_days_heard) / NULLIF(SUM(numbered_weeks.weekend_days), 0))
        / NULLIF(SUM(numbered_weeks.midweek_days_heard) / NULLIF(SUM(numbered_weeks.midweek_days), 0), 0)
            AS detection_day_ratio
    FROM resample_draws
    JOIN numbered_weeks ON numbered_weeks.week_number = resample_draws.week_number
    GROUP BY resample_draws.replicate
)

SELECT
    (SELECT MAX(total_weeks) FROM numbered_weeks) AS weeks,
    ROUND(PERCENTILE_CONT(0.025) WITHIN GROUP (ORDER BY detection_day_ratio), 2) AS range_low,
    ROUND(PERCENTILE_CONT(0.975) WITHIN GROUP (ORDER BY detection_day_ratio), 2) AS range_high
FROM resampled_ratios

## The Public Ratio Over the Same Days
The public weekend ratio inside the hydrophone window, so the two are compared over the same days.

In [ ]:
%%sql
SELECT
    COUNT(*) AS public_records,
    ROUND(
        (COUNT_IF(dim_date.is_weekend) / 2) / NULLIF(COUNT_IF(dim_date.is_midweek) / 3, 0),
        2
    ) AS weekend_ratio
FROM fct_sighting
JOIN dim_location ON dim_location.location_key = fct_sighting.location_key
JOIN dim_date ON dim_date.date_key = fct_sighting.date_key
JOIN dim_dataset ON dim_dataset.dataset_key = fct_sighting.dataset_key
WHERE dim_location.in_bay
  AND dim_dataset.kind = 'public'
  AND dim_date.in_hydrophone_window

## Result
A weekend day carries **1.52×** the public sightings of a Tuesday-to-Thursday day (95% range 1.42 to 1.63,
from 1,106 resampled weeks), and 1.37× those of an average weekday. For the hydrophones, a weekend day was
0.97× as likely as a midweek day to have a whale heard (95% range 0.94 to 1.01), while over the same 941 days
a weekend day still carried 1.31× the public sightings of a midweek day. Ship and aircraft surveys show no
weekend rise either: a weekend day carries 0.44× the survey records of a midweek day.

## Limits
* The hydrophones heard whales on 858 of the 941 days in their window, so their ratio has little room to
  rise. The window is calendar days, so a day they were out of the water counts as not heard, and weekend
  boat noise could hide some calls.
* The survey figures rest on about 150 survey days from a handful of programmes, mostly 1980s flights.
  "Other" is mostly two weekday-only shore studies of the gray whale migration, in 2014 and 2015.
* Each record counts as one sighting, though one boat trip can log several.